<a href="https://colab.research.google.com/github/luciazarpe/TFM/blob/main/preprocesamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from datetime import timedelta
import numpy as np
import plotly.express as px

import warnings
warnings.filterwarnings("ignore")

import holidays
import gc

In [2]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [3]:
try:
  from google.colab import drive
  drive.mount('/content/drive')
  DATA_PATH = '/content/drive/MyDrive/data_dsmarket/'
except ImportError:
    DATA_PATH = 'data_dsmarket/'


df_daily = pd.read_csv(DATA_PATH + 'daily_calendar_with_events.csv')
df_prices = pd.read_csv(DATA_PATH + 'item_prices.csv')
df_sales = pd.read_csv(DATA_PATH + 'item_sales.csv')


In [4]:
def reduce_mem_usage(df, turn_cat = False, silence = True):
    """Itera sobre todo el dataset convirtiendo cada columna en el tipo más adecuado para ahorrar memoria
    Parameters
    ----------
    df : pd.DataFrame
        Dataframe que se quiere reducir
    turn_cat : bool, optional
        Transformación de las columnas objeto o string a category, by default False
    Returns
    -------
    pd.DataFrame
        Dataframe optimizado
    """
    start_mem = df.memory_usage().sum() / 1024**2
    # print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))
    for col in df.columns:
        col_type = df[col].dtype

        # Saltar columnas que ya son category o datetime
        if str(col_type) in ('category', 'datetime64[ns]'):
            continue

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                # Desactivamos el casteo a float16 por los errores de precisión
                # https://github.com/pandas-dev/pandas/issues/34618
                # if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                #     df[col] = df[col].astype(np.float16)
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        if turn_cat:
            df[col] = df[col].astype('category')
    end_mem = df.memory_usage().sum() / 1024**2
    # print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    if not silence:
        print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    return df

In [5]:
df_daily  = reduce_mem_usage(df_daily, silence=False)
df_prices = reduce_mem_usage(df_prices, silence=False)
df_sales  = reduce_mem_usage(df_sales, silence=False)

Decreased by 17.5%
Decreased by 20.0%
Decreased by 78.7%


# **Exploración inicial**

## **Tabla Ventas**

In [6]:
print(f'{df_sales.shape[0]:,} filas x {df_sales.shape[1]:,} columnas')

30,490 filas x 1,920 columnas


In [7]:
display(df_sales.head())
# df_sales.describe(include= 'all')

,id,item,category,department,store,store_code,region,d_1,d_2,d_3,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,ACCESORIES_1_001_NYC_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,ACCESORIES_1_002_NYC_1,ACCESORIES_1_002,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,ACCESORIES_1_003_NYC_1,ACCESORIES_1_003,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,ACCESORIES_1_004_NYC_1,ACCESORIES_1_004,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,ACCESORIES_1_005_NYC_1,ACCESORIES_1_005,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,2,1,1,0,1,1,2,2,2,4


In [8]:
df_sales.isna().sum().sum()      # No hay nulos

0

In [9]:
df_sales.dtypes.head(10)

id            object
item          object
category      object
department    object
store         object
store_code    object
region        object
d_1            int16
d_2            int16
d_3            int16
dtype: object

In [10]:
df_sales.category.value_counts()

SUPERMARKET      14370
HOME_&_GARDEN    10470
ACCESORIES        5650
Name: category, dtype: int64

In [11]:
df_sales.department.value_counts().sort_index()

ACCESORIES_1       4160
ACCESORIES_2       1490
HOME_&_GARDEN_1    5320
HOME_&_GARDEN_2    5150
SUPERMARKET_1      2160
SUPERMARKET_2      3980
SUPERMARKET_3      8230
Name: department, dtype: int64

In [12]:
df_sales.store_code.value_counts()

NYC_1    3049
NYC_2    3049
NYC_3    3049
NYC_4    3049
BOS_1    3049
BOS_2    3049
BOS_3    3049
PHI_1    3049
PHI_2    3049
PHI_3    3049
Name: store_code, dtype: int64

In [13]:
df_sales.store.value_counts()

Greenwich_Village    3049
Harlem               3049
Tribeca              3049
Brooklyn             3049
South_End            3049
Roxbury              3049
Back_Bay             3049
Midtown_Village      3049
Yorktown             3049
Queen_Village        3049
Name: store, dtype: int64

In [14]:
df_sales.region.value_counts()

New York        12196
Boston           9147
Philadelphia     9147
Name: region, dtype: int64

In [15]:
df_sales.groupby('store')['item'].count()

store
Back_Bay             3049
Brooklyn             3049
Greenwich_Village    3049
Harlem               3049
Midtown_Village      3049
Queen_Village        3049
Roxbury              3049
South_End            3049
Tribeca              3049
Yorktown             3049
Name: item, dtype: int64

El dataset contiene **3.049 productos** organizados en **3 categorías** y **7 departamentos**,
con ventas registradas en **10 tiendas** distribuidas en **3 regiones** (New York, Boston y Philadelphia).

El catálogo de productos es **idéntico en todas las tiendas**, aunque New York tiene 4 tiendas frente
a las 3 de Boston y Philadelphia.

## **Tabla Precios**

In [16]:
print(f'{df_prices.shape[0]:,} filas x {df_prices.shape[1]:,} columnas')

6,965,706 filas x 5 columnas


In [17]:
df_prices.describe()

,yearweek,sell_price
count,6721786.00,6965706.00
mean,201382.16,5.52
std,7072.17,4.34
min,201105.00,0.01
25%,201248.00,2.62
50%,201410.00,4.20
75%,201515.00,7.18
max,201617.00,134.15


In [18]:
df_prices = df_prices.sort_values(by = 'yearweek')

In [19]:
df_prices.head()

,item,category,store_code,yearweek,sell_price
5539160,SUPERMARKET_3_702,SUPERMARKET,PHI_1,201105.00,3.94
6613947,HOME_&_GARDEN_2_445,HOME_&_GARDEN,PHI_3,201105.00,8.10
261069,HOME_&_GARDEN_2_041,HOME_&_GARDEN,NYC_1,201105.00,6.59
1516539,HOME_&_GARDEN_1_117,HOME_&_GARDEN,NYC_3,201105.00,11.21
563249,SUPERMARKET_3_188,SUPERMARKET,NYC_1,201105.00,2.38


In [20]:
df_prices.isna().sum()

item               0
category           0
store_code         0
yearweek      243920
sell_price         0
dtype: int64

In [21]:
df_prices.sell_price.value_counts()

2.38     221088
3.58     217873
3.00     189541
1.20     150387
4.78     142487
          ...  
10.53         1
4.75          1
0.55          1
2.21          1
25.10         1
Name: sell_price, Length: 1888, dtype: int64

In [22]:
df_prices.sell_price.describe()

count   6965706.00
mean          5.52
std           4.35
min           0.01
25%           2.62
50%           4.20
75%           7.18
max         134.15
Name: sell_price, dtype: float64

In [23]:
df_prices.item.value_counts()

SUPERMARKET_3_702      2870
HOME_&_GARDEN_1_234    2870
ACCESORIES_1_194       2870
SUPERMARKET_1_168      2870
SUPERMARKET_3_590      2870
                       ... 
HOME_&_GARDEN_1_308     652
HOME_&_GARDEN_1_159     633
HOME_&_GARDEN_1_242     610
SUPERMARKET_3_296       602
SUPERMARKET_2_379       543
Name: item, Length: 3049, dtype: int64

Los 3049 productos tienen distinto número de registros de precio —> no todos
estuvieron disponibles en todas las tiendas durante todas las semanas

## **Tabla Eventos**

In [24]:
print(f'{df_daily.shape[0]:,} filas x {df_daily.shape[1]:,} columnas')

1,913 filas x 5 columnas


In [25]:
df_daily.columns

Index(['date', 'weekday', 'weekday_int', 'd', 'event'], dtype='object')

In [26]:
df_daily.isna().sum()

date              0
weekday           0
weekday_int       0
d                 0
event          1887
dtype: int64

In [27]:
display(df_daily.head())
df_daily.describe(include= 'all')

,date,weekday,weekday_int,d,event
0,2011-01-29,Saturday,1,d_1,NaN
1,2011-01-30,Sunday,2,d_2,NaN
2,2011-01-31,Monday,3,d_3,NaN
3,2011-02-01,Tuesday,4,d_4,NaN
4,2011-02-02,Wednesday,5,d_5,NaN


,date,weekday,weekday_int,d,event
count,1913,1913,1913.00,1913,26
unique,1913,7,NaN,1913,5
top,2011-01-29,Saturday,NaN,d_1,SuperBowl
freq,1,274,NaN,1,6
mean,NaN,NaN,4.00,NaN,NaN
std,NaN,NaN,2.00,NaN,NaN
min,NaN,NaN,1.00,NaN,NaN
25%,NaN,NaN,2.00,NaN,NaN
50%,NaN,NaN,4.00,NaN,NaN
75%,NaN,NaN,6.00,NaN,NaN


In [28]:
df_daily.dtypes

date           object
weekday        object
weekday_int      int8
d              object
event          object
dtype: object

In [29]:
df_daily['date'] = pd.to_datetime( df_daily['date'])
df_daily = df_daily.sort_values(by = 'date')
# como esta ordenado luego me tiene que funcionar directamente lo de comprobar si esta completo

In [30]:
print(df_daily['date'].min())
print(df_daily['date'].max())
print(df_daily['date'].max() - df_daily['date'].min())

2011-01-29 00:00:00
2016-04-24 00:00:00
1912 days 00:00:00


El calendario tiene 1.913 días —> comprobamos que no falta ningún día.

In [31]:
def comprobar_fechas(c):
    inicio = c.min()
    fin = c.max()
    completo = pd.date_range(inicio, fin)
    print((completo == c).all())

comprobar_fechas(df_daily['date'])

True


In [32]:
df_daily['event'].value_counts(dropna= False)


NaN               1887
SuperBowl            6
Ramadan starts       5
Thanksgiving         5
NewYear              5
Easter               5
Name: event, dtype: int64

La gran mayoría de días no tienen ningún evento registrado — solo 26 días de los 1.913 tienen evento.

In [33]:
display(df_daily[df_daily['event'] == 'Ramadan starts'])
display(df_daily[df_daily['event'] == 'SuperBowl'])
display(df_daily[df_daily['event'] == 'Thanksgiving'])
display(df_daily[df_daily['event'] == 'NewYear'])
display(df_daily[df_daily['event'] == 'Easter'])


,date,weekday,weekday_int,d,event
184,2011-08-01,Monday,3,d_185,Ramadan starts
538,2012-07-20,Friday,7,d_539,Ramadan starts
892,2013-07-09,Tuesday,4,d_893,Ramadan starts
1247,2014-06-29,Sunday,2,d_1248,Ramadan starts
1601,2015-06-18,Thursday,6,d_1602,Ramadan starts


,date,weekday,weekday_int,d,event
8,2011-02-06,Sunday,2,d_9,SuperBowl
372,2012-02-05,Sunday,2,d_373,SuperBowl
736,2013-02-03,Sunday,2,d_737,SuperBowl
1100,2014-02-02,Sunday,2,d_1101,SuperBowl
1464,2015-02-01,Sunday,2,d_1465,SuperBowl
1835,2016-02-07,Sunday,2,d_1836,SuperBowl


,date,weekday,weekday_int,d,event
299,2011-11-24,Thursday,6,d_300,Thanksgiving
663,2012-11-22,Thursday,6,d_664,Thanksgiving
1034,2013-11-28,Thursday,6,d_1035,Thanksgiving
1398,2014-11-27,Thursday,6,d_1399,Thanksgiving
1762,2015-11-26,Thursday,6,d_1763,Thanksgiving


,date,weekday,weekday_int,d,event
337,2012-01-01,Sunday,2,d_338,NewYear
703,2013-01-01,Tuesday,4,d_704,NewYear
1068,2014-01-01,Wednesday,5,d_1069,NewYear
1433,2015-01-01,Thursday,6,d_1434,NewYear
1798,2016-01-01,Friday,7,d_1799,NewYear


,date,weekday,weekday_int,d,event
435,2012-04-08,Sunday,2,d_436,Easter
792,2013-03-31,Sunday,2,d_793,Easter
1177,2014-04-20,Sunday,2,d_1178,Easter
1527,2015-04-05,Sunday,2,d_1528,Easter
1884,2016-03-27,Sunday,2,d_1885,Easter


Cada evento aparece una vez por año excepto SuperBowl, que al caer en febrero
aparece tanto en 2011 como en 2016 al ser los extremos del dataset.

Las fechas de cada evento pueden variar un poco año a año (Easter, Thanksgiving...) excepto NewYear, que siempre cae el 1 de enero.

In [34]:
(df_daily['d'].str[2:].astype('int') == pd.Series(range(1,1914))).all()
# la columna d son los numeros normales ordenados y no falta ninguno.

True

In [35]:
df_daily.groupby('weekday')['weekday_int'].value_counts()

weekday    weekday_int
Friday     7              273
Monday     3              273
Saturday   1              274
Sunday     2              274
Thursday   6              273
Tuesday    4              273
Wednesday  5              273
Name: weekday_int, dtype: int64

Vemos que es información redundante, por lo que eliminamos la columna 'weelday_int':

In [36]:
df_daily = df_daily.drop(columns=['weekday_int'])

In [37]:
def yearweek(dt):
    offsetdt = dt + timedelta(days=2)
    return offsetdt.strftime("%Y%W")

df_daily['yearweek'] = df_daily['date'].apply(lambda x: yearweek(x)).astype(float)

df_daily

,date,weekday,d,event,yearweek
0,2011-01-29,Saturday,d_1,NaN,201105.00
1,2011-01-30,Sunday,d_2,NaN,201105.00
2,2011-01-31,Monday,d_3,NaN,201105.00
3,2011-02-01,Tuesday,d_4,NaN,201105.00
4,2011-02-02,Wednesday,d_5,NaN,201105.00
...,...,...,...,...,...
1908,2016-04-20,Wednesday,d_1909,NaN,201616.00
1909,2016-04-21,Thursday,d_1910,NaN,201616.00
1910,2016-04-22,Friday,d_1911,NaN,201616.00
1911,2016-04-23,Saturday,d_1912,NaN,201617.00


### Añadir eventos

In [38]:
us_holidays = holidays.US(years=range(2011, 2017))

# Días festivos que cubre la librería en nuestro rango de fechas
festivos_df = pd.DataFrame([{'date': fecha, 'holiday': nombre}
                            for fecha, nombre in us_holidays.items()
                            ]).sort_values('date')

In [39]:
print("Eventos que ya tenemos:")
print(df_daily['event'].value_counts(dropna=False))

print("\nEventos que añade la librería:")
print(festivos_df['holiday'].value_counts())

Eventos que ya tenemos:
NaN               1887
SuperBowl            6
Ramadan starts       5
Thanksgiving         5
NewYear              5
Easter               5
Name: event, dtype: int64

Eventos que añade la librería:
New Year's Day                 6
Martin Luther King Jr. Day     6
Washington's Birthday          6
Memorial Day                   6
Independence Day               6
Labor Day                      6
Columbus Day                   6
Veterans Day                   6
Thanksgiving Day               6
Christmas Day                  6
Christmas Day (observed)       2
New Year's Day (observed)      1
Veterans Day (observed)        1
Independence Day (observed)    1
Name: holiday, dtype: int64


Añadimos festivos oficiales de EEUU con la librería `holidays`.
La librería incluye festivos que ya teníamos (***NewYear***, ***Thanksgiving***) y versiones
***observed*** (días de sustitución cuando el festivo cae en fin de semana)



La ***máscara isna()*** protege los eventos que ya teníamos para no sobrescribirlos


In [40]:
us_holidays = holidays.US(years=range(2011, 2017))

def get_holiday(fecha):
    return us_holidays.get(fecha)

mask = df_daily['event'].isna()         # Para no sobreescribir los eventos que ya tenemos
df_daily.loc[mask, 'event'] = df_daily.loc[mask, 'date'].apply(get_holiday)

df_daily['event'].value_counts(dropna=False)

None                           1842
Washington's Birthday             6
SuperBowl                         6
Veterans Day                      5
Easter                            5
Martin Luther King Jr. Day        5
NewYear                           5
Christmas Day                     5
Thanksgiving                      5
Columbus Day                      5
Labor Day                         5
Ramadan starts                    5
Independence Day                  5
Memorial Day                      5
Christmas Day (observed)          1
New Year's Day (observed)         1
Veterans Day (observed)           1
Independence Day (observed)       1
Name: event, dtype: int64

In [41]:
eventos_extra = {
    # climaticos
    '2012-10-29': 'Hurricane Sandy',
    '2012-10-30': 'Hurricane Sandy',
    '2015-01-26': 'Blizzard 2015',
    '2015-01-27': 'Blizzard 2015',
    '2016-01-23': 'Blizzard 2016',
    '2016-01-24': 'Blizzard 2016',
    # otros
    '2011-11-25': 'Black Friday', '2012-11-23': 'Black Friday',
    '2013-11-29': 'Black Friday', '2014-11-28': 'Black Friday',
    '2015-11-27': 'Black Friday', '2016-11-25': 'Black Friday',
    '2011-11-28': 'Cyber Monday', '2012-11-26': 'Cyber Monday',
    '2013-12-02': 'Cyber Monday', '2014-12-01': 'Cyber Monday',
    '2015-11-30': 'Cyber Monday', '2016-11-28': 'Cyber Monday',
    # sn valentin
    '2011-02-14': "Valentine's Day", '2012-02-14': "Valentine's Day",
    '2013-02-14': "Valentine's Day", '2014-02-14': "Valentine's Day",
    '2015-02-14': "Valentine's Day", '2016-02-14': "Valentine's Day",
}

mask_extra = df_daily['date'].dt.strftime('%Y-%m-%d').isin(eventos_extra.keys())
mask_sin_evento = df_daily['event'].isna()

df_daily.loc[mask_extra & mask_sin_evento, 'event'] = df_daily.loc[mask_extra & mask_sin_evento, 'date'].dt.strftime('%Y-%m-%d').map(eventos_extra)

print(df_daily['event'].value_counts(dropna=False))

None                           1820
Valentine's Day                   6
Washington's Birthday             6
SuperBowl                         6
Thanksgiving                      5
Easter                            5
Martin Luther King Jr. Day        5
NewYear                           5
Christmas Day                     5
Black Friday                      5
Cyber Monday                      5
Veterans Day                      5
Columbus Day                      5
Labor Day                         5
Ramadan starts                    5
Independence Day                  5
Memorial Day                      5
Hurricane Sandy                   2
Blizzard 2015                     2
Blizzard 2016                     2
Christmas Day (observed)          1
New Year's Day (observed)         1
Veterans Day (observed)           1
Independence Day (observed)       1
Name: event, dtype: int64


# Cruzamos datasets

Las tres tablas están separadas y necesitamos unirlas en un único `df`. Lo hacemos en tres pasos:

1. **Melt df_sales ->** transformamos las ventas de formato ancho (una columna por día) a largo
(una fila por día). Resultado: ~58M filas.

2. **Merge 1: Ventas + Calendario ->** añadimos a cada registro de venta la fecha real,
el día de la semana y el evento de ese día, usando la columna `d` como clave.

3. **Merge 2: + Precios ->** añadimos el precio de cada producto en cada tienda y semana,
usando `item + store_code + yearweek` como clave. Si no hay precio registrado esa semana,
el campo `sell_price` queda como NaN.

In [42]:
# Mantenemos el nivel de granularidad más bajo posible: una fila por producto × día × tienda.
# La predicción final es diaria, por lo que agregar a semanal supondría perder información.
# La columna yearweek solo se usa como clave de unión con la tabla de precios (que son semanales), nunca para agregar ventas.


# Paso 1:
df_sales_melt = pd.melt(df_sales, id_vars=df_sales.columns[:7], var_name='d', value_name='sales')

# Strings como categóricas para ahorrar memoria
for col in ['item', 'id', 'store_code', 'store', 'region', 'category', 'd']:
    df_sales_melt[col] = df_sales_melt[col].astype('category')

for col in ['item', 'store_code']:
    df_prices[col] = df_prices[col].astype('category')


# Paso 2:
df = df_sales_melt.merge(df_daily, on='d', how='left')

# # Paso 3:
# df = df.merge(df_prices.drop(columns='category'), on=['item', 'store_code', 'yearweek'], how='left')

## Control de demanada intermitente

Construimos un dataset en el que tenemso por cada producto y tienda cuando se vende por primera vez, cuando se vende por última vez y cuántas semanas se vende en total. Esto nos permitirá detectar productos que aparecen tarde o desaparecen pronto, lo que puede ser un problema para el modelo.

In [43]:
todas_las_semanas = sorted(df_prices['yearweek'].dropna().astype(int).unique())
semana_a_posicion = {semana: i for i, semana in enumerate(todas_las_semanas)}

def huecos_internos_semanas_reales(grupo):
    semanas = sorted(grupo['yearweek'].dropna().astype(int).unique())
    
    if not semanas:
        return pd.Series({
            'primera_semana': pd.NA,
            'ultima_semana': pd.NA,
            'semanas_observadas': 0,
            'semanas_esperadas_en_tramo': 0,
            'semanas_faltantes_internas': 0
        })
    
    primera_semana = semanas[0]
    ultima_semana = semanas[-1]
    semanas_observadas = len(semanas)
    
    inicio = semana_a_posicion[primera_semana]
    fin = semana_a_posicion[ultima_semana]
    semanas_esperadas_en_tramo = fin - inicio + 1
    
    semanas_faltantes_internas = semanas_esperadas_en_tramo - semanas_observadas
    
    return pd.Series({
        'primera_semana': primera_semana,
        'ultima_semana': ultima_semana,
        'semanas_observadas': semanas_observadas,
        'semanas_esperadas_en_tramo': semanas_esperadas_en_tramo,
        'semanas_faltantes_internas': semanas_faltantes_internas
    })

comprobacion_huecos = (
    df_prices
    .groupby(['item', 'store_code'])
    .apply(huecos_internos_semanas_reales)
    .reset_index()
)

comprobacion_huecos.head()

,item,store_code,primera_semana,ultima_semana,semanas_observadas,semanas_esperadas_en_tramo,semanas_faltantes_internas
0,ACCESORIES_1_001,BOS_1,201328,201617,149,149,0
1,ACCESORIES_1_001,BOS_2,201330,201617,147,147,0
2,ACCESORIES_1_001,BOS_3,201330,201617,147,147,0
3,ACCESORIES_1_001,NYC_1,201328,201617,149,149,0
4,ACCESORIES_1_001,NYC_2,201330,201617,147,147,0


para cada producto mira que semanas tiene, cual es la primera y la ulitma semana, cuantas deberia haber entre esa fecha de inicio y de final y cuantas faltan realmente entre esas semanas.

In [44]:
semanas_maestras = set(df['yearweek'].dropna().astype(int).unique())
semanas_prices = set(df_prices['yearweek'].dropna().astype(int).unique())

semanas_que_faltan_en_prices = sorted(semanas_maestras - semanas_prices)
semanas_que_sobran_en_prices = sorted(semanas_prices - semanas_maestras)

print("Semanas en df =", len(semanas_maestras))
print("Semanas en df_prices =", len(semanas_prices))
print("Semanas que faltan en df_prices =", len(semanas_que_faltan_en_prices))
print("Semanas que sobran en df_prices =", len(semanas_que_sobran_en_prices))

if len(semanas_que_faltan_en_prices) == 0:
    print("Sí, df_prices tiene todas las semanas de df.")
else:
    print("No, faltan estas semanas en df_prices:")
    print(semanas_que_faltan_en_prices)

if len(semanas_que_sobran_en_prices) > 0:
    print("Además, estas semanas están en df_prices pero no en df:")
    print(semanas_que_sobran_en_prices)

Semanas en df = 279
Semanas en df_prices = 279
Semanas que faltan en df_prices = 0
Semanas que sobran en df_prices = 0
Sí, df_prices tiene todas las semanas de df.


In [45]:
comprobacion_huecos[ comprobacion_huecos['semanas_faltantes_internas'] > 0 ]

,item,store_code,primera_semana,ultima_semana,semanas_observadas,semanas_esperadas_en_tramo,semanas_faltantes_internas


In [46]:
# vamos a ver si hay productos que no aparecen la primera semana 
comprobacion_huecos[ comprobacion_huecos['primera_semana'] != df_prices['yearweek'].min() ]['item'].count()

19558

In [47]:
# ahora vamos a ver si hay productos que desaparecen antes de la iltima semana
comprobacion_huecos[ comprobacion_huecos['ultima_semana'] != df_prices['yearweek'].max() ]['item'].count()

0

por ahora sabemos que hay productos que entran al catalogo mas tarde pero no que desaparezcan antes ni tengan demanda intermitente 

In [48]:
filas_con_yearweek_nulo = df_prices[df_prices['yearweek'].isna()]
filas_con_yearweek_nulo

,item,category,store_code,yearweek,sell_price
149,ACCESORIES_1_001,ACCESORIES,NYC_1,NaN,11.15
150,ACCESORIES_1_001,ACCESORIES,NYC_1,NaN,11.15
151,ACCESORIES_1_001,ACCESORIES,NYC_1,NaN,11.15
152,ACCESORIES_1_001,ACCESORIES,NYC_1,NaN,11.15
153,ACCESORIES_1_001,ACCESORIES,NYC_1,NaN,11.15
...,...,...,...,...,...
6965701,SUPERMARKET_3_827,SUPERMARKET,PHI_3,NaN,1.20
6965702,SUPERMARKET_3_827,SUPERMARKET,PHI_3,NaN,1.20
6965703,SUPERMARKET_3_827,SUPERMARKET,PHI_3,NaN,1.20
6965704,SUPERMARKET_3_827,SUPERMARKET,PHI_3,NaN,1.20


In [49]:
semanas_maestras = set(df['yearweek'].dropna().astype(int).unique())
semanas_prices = set(df_prices['yearweek'].dropna().astype(int).unique())

semanas_que_faltan_en_prices = sorted(semanas_maestras - semanas_prices)

print("Semanas que faltan en df_prices respecto a df =", semanas_que_faltan_en_prices)

Semanas que faltan en df_prices respecto a df = []


In [50]:
# vamos a unir los precios
df = df.merge(
    df_prices[['item', 'store_code', 'yearweek', 'sell_price']],
    on = ['item', 'store_code', 'yearweek'],
    how = 'left'
)
# para q no vuelva a coger la cagteogia q sale en las dos tablas

In [52]:
primera_semana_precio = (
    df_prices[df_prices['sell_price'].notna()]
    .groupby(['item', 'store_code'])['yearweek']
    .min()
    .reset_index()
    .rename(columns = {'yearweek': 'primera_semana_precio'})
)

df_final = df.merge(
    primera_semana_precio,
    on = ['item', 'store_code'],
    how = 'left'
)


In [53]:
df_final

,id,item,category,department,store,store_code,region,d,sales,date,weekday,event,yearweek,sell_price,primera_semana_precio
0,ACCESORIES_1_001_NYC_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201105.00,NaN,201328.00
1,ACCESORIES_1_002_NYC_1,ACCESORIES_1_002,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201105.00,NaN,201125.00
2,ACCESORIES_1_003_NYC_1,ACCESORIES_1_003,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201105.00,NaN,201405.00
3,ACCESORIES_1_004_NYC_1,ACCESORIES_1_004,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201105.00,NaN,201110.00
4,ACCESORIES_1_005_NYC_1,ACCESORIES_1_005,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201105.00,NaN,201121.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58327365,SUPERMARKET_3_823_PHI_3,SUPERMARKET_3_823,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,1,2016-04-24,Sunday,None,201617.00,3.58,201105.00
58327366,SUPERMARKET_3_824_PHI_3,SUPERMARKET_3_824,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,0,2016-04-24,Sunday,None,201617.00,2.98,201105.00
58327367,SUPERMARKET_3_825_PHI_3,SUPERMARKET_3_825,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,0,2016-04-24,Sunday,None,201617.00,4.78,201105.00
58327368,SUPERMARKET_3_826_PHI_3,SUPERMARKET_3_826,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,3,2016-04-24,Sunday,None,201617.00,1.54,201334.00


In [54]:
df_final['nulo_antes_de_aparecer'] = (
    df_final['sell_price'].isna() &
    (df_final['yearweek'] < df_final['primera_semana_precio'])
)

df_final['nulo_despues_de_aparecer'] = (
    df_final['sell_price'].isna() &
    (df_final['yearweek'] >= df_final['primera_semana_precio'])
)

In [55]:
print("Nulos antes de aparecer =", df_final['nulo_antes_de_aparecer'].sum())
print("Nulos después de aparecer =", df_final['nulo_despues_de_aparecer'].sum())

Nulos antes de aparecer = 12299413
Nulos después de aparecer = 0


Comprobamos que `ACCESORIES_1_001` en `BOS_1` tiene NaN antes de la semana `201328`, que es cuando empieza a existir ese producto en esa tienda.

In [58]:
primera_semana_precio[ (primera_semana_precio['item'] == 'ACCESORIES_1_001') & (primera_semana_precio['store_code'] == 'BOS_1') ]['primera_semana_precio'].values[0]

201328.0

In [59]:
df[(df['item'] == 'ACCESORIES_1_001') & (df['store_code'] == 'BOS_1')][['date', 'yearweek', 'sell_price']].sort_values('date')

,date,yearweek,sell_price
12196,2011-01-29,201105.00,NaN
42686,2011-01-30,201105.00,NaN
73176,2011-01-31,201105.00,NaN
103666,2011-02-01,201105.00,NaN
134156,2011-02-02,201105.00,NaN
...,...,...,...
58187116,2016-04-20,201616.00,10.99
58217606,2016-04-21,201616.00,10.99
58248096,2016-04-22,201616.00,10.99
58278586,2016-04-23,201617.00,10.99


## Eliminación de filas anteriores a la activación del producto

Las filas con `sell_price = NaN` corresponden a registros anteriores a la existencia del producto en esa tienda. Vamos a comprobar que todas tienen `sales = 0`, por lo que no contienen información real — el 0 no representa demanda intermitente sino ausencia del producto.

Vamos a eliminar estas filas para que el modelo no confunda ambos tipos de 0.

In [60]:
# Comprobamos que todas las filas sin precio tienen ventas = 0
print(df[df['sell_price'].isna()]['sales'].value_counts())

0    12299413
Name: sales, dtype: int64


In [61]:
print(f"Filas eliminadas: {df['sell_price'].isna().sum():,}")
print(f"Filas restantes: {df['sell_price'].notna().sum():,}")

Filas eliminadas: 12,299,413
Filas restantes: 46,027,957


In [62]:
df = df[df['sell_price'].notna()].reset_index(drop=True)

In [63]:
# Verificamos el estado final de nulos tras la eliminación
df.isna().sum()

id                   0
item                 0
category             0
department           0
store                0
store_code           0
region               0
d                    0
sales                0
date                 0
weekday              0
event         43763326
yearweek             0
sell_price           0
dtype: int64

# Creación variables auxiliares
Añadimos algunas variables que servirán como features para la modelización

In [64]:
def estacion(mes):
    if mes in [12, 1, 2]:
        return 'Invierno'
    elif mes in [3, 4, 5]:
        return 'Primavera'
    elif mes in [6, 7, 8]:
        return 'Verano'
    else:
        return 'Otoño'

df['season'] = df['date'].dt.month.apply(estacion)

In [65]:
# En EEUU lo más habitual es cobrar dos veces al mes: en torno al día 1 y al día 15.
# Creamos ventanas de ±2 días alrededor de cada fecha de cobro para intentar capturar el efecto en el comportamiento de compra.
def pay_period(dia):
    if dia <= 3:
        return 'inicio_mes'
    elif 14 <= dia <= 16:
        return 'mitad_mes'
    else:
        return 'otros'

df['pay_period'] = df['date'].dt.day.apply(pay_period)

In [66]:
# Creamos una columna con el precio multiplicado por las ventas para tener la facturación
df['ingresos'] = df['sales'] * df['sell_price']

df.head()

,id,item,category,department,store,store_code,region,d,sales,date,weekday,event,yearweek,sell_price,season,pay_period,ingresos
0,ACCESORIES_1_008_NYC_1,ACCESORIES_1_008,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,12,2011-01-29,Saturday,None,201105.00,0.61,Invierno,otros,7.34
1,ACCESORIES_1_009_NYC_1,ACCESORIES_1_009,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,2,2011-01-29,Saturday,None,201105.00,2.07,Invierno,otros,4.15
2,ACCESORIES_1_010_NYC_1,ACCESORIES_1_010,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201105.00,4.22,Invierno,otros,0.00
3,ACCESORIES_1_012_NYC_1,ACCESORIES_1_012,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201105.00,7.95,Invierno,otros,0.00
4,ACCESORIES_1_015_NYC_1,ACCESORIES_1_015,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,4,2011-01-29,Saturday,None,201105.00,0.93,Invierno,otros,3.72


In [67]:
# Marcamos un binario para ver si hay festivo o no en el día y poder hacer clustering
df['is_holiday'] = df['event'].notna().astype(int)

In [68]:
df

,id,item,category,department,store,store_code,region,d,sales,date,weekday,event,yearweek,sell_price,season,pay_period,ingresos,is_holiday
0,ACCESORIES_1_008_NYC_1,ACCESORIES_1_008,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,12,2011-01-29,Saturday,None,201105.00,0.61,Invierno,otros,7.34,0
1,ACCESORIES_1_009_NYC_1,ACCESORIES_1_009,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,2,2011-01-29,Saturday,None,201105.00,2.07,Invierno,otros,4.15,0
2,ACCESORIES_1_010_NYC_1,ACCESORIES_1_010,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201105.00,4.22,Invierno,otros,0.00,0
3,ACCESORIES_1_012_NYC_1,ACCESORIES_1_012,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,None,201105.00,7.95,Invierno,otros,0.00,0
4,ACCESORIES_1_015_NYC_1,ACCESORIES_1_015,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,4,2011-01-29,Saturday,None,201105.00,0.93,Invierno,otros,3.72,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46027952,SUPERMARKET_3_823_PHI_3,SUPERMARKET_3_823,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,1,2016-04-24,Sunday,None,201617.00,3.58,Primavera,otros,3.58,0
46027953,SUPERMARKET_3_824_PHI_3,SUPERMARKET_3_824,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,0,2016-04-24,Sunday,None,201617.00,2.98,Primavera,otros,0.00,0
46027954,SUPERMARKET_3_825_PHI_3,SUPERMARKET_3_825,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,0,2016-04-24,Sunday,None,201617.00,4.78,Primavera,otros,0.00,0
46027955,SUPERMARKET_3_826_PHI_3,SUPERMARKET_3_826,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,3,2016-04-24,Sunday,None,201617.00,1.54,Primavera,otros,4.61,0


In [70]:
# vamos a meter un indice estacional para ver si hay productos que tienen un comportamiento muy diferente en semanas concretas
df['yearweek'] = df['yearweek'].astype(int).astype(str)
df['year'] = df['yearweek'].str[:4].astype(int)
df['week'] = df['yearweek'].str[4:].astype(int)
df['month'] = df['date'].dt.month

df_semanal = df.groupby(['item', 'store_code', 'year', 'week']).agg({'sales': 'sum', 'ingresos': 'sum'})
df_semanal = df_semanal.reset_index()
media_anual = df_semanal.groupby(['item', 'year', 'store_code'], as_index=False).agg(media_semanal_anual=('ingresos', 'mean'))
df_semanal = df_semanal.merge(media_anual, on=['item', 'year', 'store_code'])
df_semanal['indice_estacionalidad'] = df_semanal['ingresos'] / df_semanal['media_semanal_anual']
df_semanal = df_semanal.replace([np.inf, -np.inf], np.nan)
df_semanal = df_semanal.dropna(subset=['indice_estacionalidad'])

indice_tienda_producto = df_semanal.groupby(['store_code', 'item', 'week'], as_index=False).agg(indice_estacional_store_item=('indice_estacionalidad', 'mean'))
df = df.merge(indice_tienda_producto, on=['store_code', 'item', 'week'], how='left')

# Guardar dataset preprocesado

Exportamos el dataframe final a formato feather, que preserva los tipos de datos optimizados y sirve como punto de partida para el EDA.

In [71]:
df = df.reset_index(drop=True)

df.to_feather(DATA_PATH + 'df_preprocessed.feather')